# 描述器与元类

学习目标：解释属性读取与赋值如何触发描述器，编写带校验的属性，并根据需要选择 slots、类创建钩子、类装饰器或元类。

前置知识：类与实例、实例字典、继承与 MRO、super、property、装饰器、异常处理和类型标注。

运行环境：Python 3.12。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

## 1 普通属性先看实例，再看类

同一种记录可以有默认标题，单个实例也可以保存自己的标题。没有描述器介入时，实例字典中的同名值会遮蔽类属性；删除实例中的值后，又能读到类属性。

MRO 是方法解析顺序，也用于寻找类层次中的属性。这里先观察普通属性；下一节会加入描述器对读取优先级的影响。

In [1]:
class Record:
    """为记录提供默认标题。"""

    title = "未命名"


class LessonRecord(Record):
    """继承记录的默认标题。"""


record = LessonRecord()
print(record.title)  # 未命名：类层次中继承的默认值。
record.title = "属性查找"
print(vars(record))  # {'title': '属性查找'}：写入实例字典。
print(Record.title)  # 未命名：实例赋值没有修改类属性。
del record.title
print(record.title)  # 未命名：实例项删除后重新读到类属性。
print([base.__name__ for base in LessonRecord.__mro__])
# ['LessonRecord', 'Record', 'object']：类层次按此顺序查找。

未命名
{'title': '属性查找'}
未命名
未命名
['LessonRecord', 'Record', 'object']


## 2 描述器参与属性读取和赋值

描述器（descriptor）是实现属性访问协议的对象。把它放在所属类或基类的命名空间中，普通属性访问就能触发相应方法；仅把描述器对象放进实例字典，不会自动触发这一机制。

实例字典已经放入同名值，读取时为什么仍可能得到描述器的结果？先按下面的优先级判断当前命中哪一层。

![默认实例属性读取：按优先级寻找结果](image/illustration/31-01-descriptor-precedence.svg)

图示：默认属性读取次序示意；并非逐层执行所有 getter。先沿 MRO 找到的类成员决定描述器分支。

| 方法 | 中文名称／含义 |
| --- | --- |
| \_\_get\_\_ | 读取属性；instance 是当前实例，类访问时为 None，owner 是访问所经的类 |
| \_\_set\_\_ | 为实例写入属性值 |
| \_\_delete\_\_ | 删除实例的受管理属性 |
| \_\_set\_name\_\_ | 类创建时获知所属类和绑定名称；它本身不构成描述器协议 |

只实现 \_\_get\_\_ 的对象是非数据描述器（non-data descriptor）。其类型还定义 \_\_set\_\_ 或 \_\_delete\_\_ 时，就是数据描述器（data descriptor）；即使写入方法只抛出异常，也仍属于数据描述器。

对于采用 object.\_\_getattribute\_\_ 默认规则的实例读取，先沿 MRO 找到第一个同名类成员，再判断它：有 \_\_get\_\_ 的数据描述器优先于实例字典；实例字典优先于非数据描述器和普通类成员。数据描述器若没有 \_\_get\_\_，不能套用上述读取优先规则。

下面 board.suggested 命中实例项，而 board.fixed 命中数据描述器；将两个输出分别定位到第②层和第①层。

In [2]:
class SuggestedTitle:
    """允许实例覆盖的默认标题。"""

    def __get__(
        self, instance: object | None, owner: type | None = None
    ) -> str:
        return "建议标题"


class FixedTitle(SuggestedTitle):
    """通过拒绝写入实现只读数据描述器。"""

    def __set__(self, instance: object, value: object) -> None:
        raise AttributeError("固定标题不可赋值")


class TitleBoard:
    """对比同名实例项与两种描述器。"""

    suggested = SuggestedTitle()
    fixed = FixedTitle()


board = TitleBoard()
# 故意在实例字典放入同名值，用一次读取对比两类描述器的优先级。
vars(board).update(suggested="实例建议", fixed="实例固定值")
print(board.suggested)  # 实例建议：实例项遮蔽非数据描述器。
print(board.fixed)  # 建议标题：数据描述器的读取仍然优先。
board.local = SuggestedTitle()
print(type(board.local).__name__)  # SuggestedTitle：实例内保存的是对象。

实例建议
建议标题
SuggestedTitle


In [3]:
# 预期 AttributeError：直接观察原始异常，之后继续运行下一单元。
# 固定标题不可赋值：写入触发 __set__。
board.fixed = "替换"

AttributeError: 固定标题不可赋值

## 3 属性访问入口与缺失回退

\_\_getattribute\_\_ 是实例普通属性读取的入口。重写后应委托 object.\_\_getattribute\_\_ 或合适的父类实现，保留描述器等规则；在入口中再次读取自身属性，可能造成递归。

点号读取和 getattr 在该入口抛出 AttributeError 后，才尝试 \_\_getattr\_\_。直接调用 object.\_\_getattribute\_\_ 不会自动补上这一步。回退也可能来自 property 内部抛出的 AttributeError，因此“进入回退”不一定表示类中没有这个名称；其他异常不会因此自动进入回退。

这里的入口不是所有操作的拦截器。len 等语法或内置操作对特殊方法的隐式查找，可以绕过实例的 \_\_getattribute\_\_。

In [4]:
class LookupRecord:
    """显示普通读取、缺失回退与隐式协议调用的区别。"""

    title = "已有标题"

    def __getattribute__(self, name: str) -> object:
        print("读取入口：", name)
        # 交给基础实现继续查找，避免再次读取 self 属性而无限递归。
        return object.__getattribute__(self, name)

    def __getattr__(self, name: str) -> str:
        if name in ("alias", "unavailable"):
            return "回退标题"
        raise AttributeError(name)

    @property
    def unavailable(self) -> str:
        """模拟已定义但当前无法提供值的属性。"""
        raise AttributeError("标题尚未准备好")

    def __len__(self) -> int:
        return 1


lookup_record = LookupRecord()
print(lookup_record.title)  # 入口之后返回已有标题，不调用缺失回退。
print(getattr(lookup_record, "alias"))  # 入口失败后得到回退标题。
print(lookup_record.unavailable)  # 先显示“读取入口： unavailable”，再显示“回退标题”。

读取入口： title
已有标题
读取入口： alias
回退标题
读取入口： unavailable
回退标题


In [5]:
# 预期 AttributeError：直接观察原始异常，之后继续运行下一单元。
object.__getattribute__(lookup_record, "alias")

AttributeError: 'LookupRecord' object has no attribute 'alias'

In [6]:
print(len(lookup_record))  # 1：前面没有出现读取 __len__ 的入口日志。

1


## 4 用描述器统一校验百分数

把同一种校验放进描述器，多个字段就能复用规则。下面的 Percent 接受 0 到 100（含端点）的内置 int 或 float；这是本例的数据约定，不是所有描述器必须遵守的限制。

\_\_set\_name\_\_ 保存字段名，并生成以下划线开头的存储名称。例如 score 对应 \_score。值保存在各实例中，描述器保存字段配置；给不同字段分别创建描述器对象，避免名称信息互相覆盖。

类访问会把 None 作为 instance 传给 \_\_get\_\_。本例在这种情况下返回描述器自身，便于检查配置；实例访问则读取相应存储属性。未初始化的存储属性仍抛出 AttributeError。

In [7]:
class Percent:
    """管理 0 到 100 的内置整数或浮点数，不接受 bool。"""

    def __set_name__(self, owner: type, name: str) -> None:
        self.owner = owner
        self.public_name = name
        self.storage_name = "_" + name

    def __get__(
        self, instance: object | None, owner: type | None = None
    ) -> "Percent | int | float":
        # 通过类读取时返回描述器本身；通过实例读取时才取得该实例的数据。
        if instance is None:
            return self
        return getattr(instance, self.storage_name)

    def __set__(self, instance: object, value: object) -> None:
        # 先检查类型与范围，失败时不改动之前保存的值。
        if type(value) not in (int, float):
            raise TypeError(f"{self.public_name} 只接受内置 int 或 float")
        if not 0 <= value <= 100:
            raise ValueError(f"{self.public_name} 必须在 0 到 100 之间")
        setattr(instance, self.storage_name, value)


class Progress:
    """分别保存得分百分比和完成百分比。"""

    score = Percent()
    completion = Percent()

    def __init__(self, score: int | float, completion: int | float) -> None:
        self.score = score
        self.completion = completion


first_progress = Progress(80, 25.5)
second_progress = Progress(60, 100)
first_progress.score = 90
print(first_progress.score, second_progress.score)  # 90 60：实例值互不干扰。
print(vars(first_progress))  # {'_score': 90, '_completion': 25.5}。
print(Progress.score is vars(Progress)["score"])  # True：类访问返回描述器。
print(Progress.score.owner.__name__, Progress.score.public_name)
# Progress score：__set_name__ 获得定义处的类与名称。

90 60
{'_score': 90, '_completion': 25.5}
True
Progress score


### 4.1 明确 bool、非数值与越界输入

bool 是 int 的子类，所以只用 isinstance 判断整数，会接受 True 和 False。本例用精确类型限制排除 bool，也不接收 int、float 的子类或其他数值类型；需要扩展数值范围时，应先修改数据约定。

NaN 表示非数值，其有序比较为 False。先判断“值满足完整闭区间条件”，再整体取反，能拒绝 NaN，也能拒绝正负无穷与普通越界值。不要仅用“小于下界或大于上界”检查 NaN。

In [8]:
print(isinstance(True, int))  # True：继承关系会影响 isinstance。
invalid_percent_cases = (
    (True, TypeError),
    (False, TypeError),
    ("80", TypeError),
    (-1, ValueError),
    (101, ValueError),
    (float("nan"), ValueError),
    (float("inf"), ValueError),
    (float("-inf"), ValueError),
)
previous_score = first_progress.score
# 每次失败后都观察旧值，确认 setter 没有先写入再检查。
for invalid_value, expected_error in invalid_percent_cases:
    try:
        first_progress.score = invalid_value
    except expected_error as error:
        assert type(error) is expected_error
        assert first_progress.score == previous_score
        print(repr(invalid_value), type(error).__name__)
    else:
        raise AssertionError(f"错误接受了百分数：{invalid_value!r}")
# 前三项是 TypeError，其余是 ValueError；每次失败都保留原来的 90。

True
True TypeError
False TypeError
'80' TypeError
-1 ValueError
101 ValueError
nan ValueError
inf ValueError
-inf ValueError


### 4.2 继承与事后绑定的名称通知

\_\_set\_name\_\_ 在所属类创建时调用。继承原来的描述器不会为它重新绑定名称；描述器记录的定义类，和 \_\_get\_\_ 接收的当前访问类，是不同概念。经子类访问继承的描述器时，\_\_get\_\_ 的 owner 是子类。

类创建后再赋予描述器，不会自动调用 \_\_set\_name\_\_。确实需要这种动态绑定时，应显式完成名称通知，再使用该属性。

In [9]:
class ChildProgress(Progress):
    """继承已有的字段配置。"""


print(ChildProgress.score is Progress.score)  # True：继承同一个描述器。
print(ChildProgress.score.owner is Progress)  # True：仍记录最初的定义类。
print(ChildProgress(50, 75).score)  # 50：子类实例也能使用字段。


class LateProgress:
    """演示类创建后的显式字段绑定。"""


late_field = Percent()
LateProgress.score = late_field
print(vars(late_field))  # {}：赋值没有自动发送名称通知。
late_field.__set_name__(LateProgress, "score")
late_progress = LateProgress()
late_progress.score = 70
print(late_progress.score, vars(late_progress))  # 70 {'_score': 70}。

True
True
50
{}
70 {'_score': 70}


## 5 property 与普通方法也使用描述器

property 是数据描述器，即使没有提供 setter，也不会变成非数据描述器。同名实例字典项不能遮蔽它；正常赋值会因为缺少 setter 而失败。

类中的普通 Python 函数是非数据描述器。经实例访问时，它返回绑定方法，把实例作为首个参数传入；绑定方法的 \_\_self\_\_ 指向实例，\_\_func\_\_ 指向原函数。实例字典可以遮蔽这个方法。

property 与方法在前面的类基础中已经使用过；这里关注它们为何具有不同的覆盖行为。

In [10]:
class Magazine:
    """提供只读标题和可被单个实例覆盖的呈现方法。"""

    def __init__(self, title: str) -> None:
        self._title = title

    @property
    def title(self) -> str:
        """返回创建时保存的标题。"""
        return self._title

    def render(self) -> str:
        """把标题用于正文预览。"""
        return "正文：" + self.title


magazine = Magazine("描述器")
vars(magazine)["title"] = "字典中的同名值"
print(magazine.title)  # 描述器：property 的读取优先。
# 分别查看绑定方法保存的实例与原函数，再用实例属性遮蔽该方法。
bound_render = magazine.render
print(bound_render.__self__ is magazine)  # True：方法绑定了实例。
print(bound_render.__func__ is vars(Magazine)["render"])  # True：原函数。
magazine.render = lambda: "临时预览"
print(magazine.render())  # 临时预览：实例中的函数没有再次自动绑定。
del magazine.render
print(magazine.render())  # 正文：描述器；删除实例项后恢复类方法。

描述器
True
True
临时预览
正文：描述器


In [11]:
# 预期 AttributeError：直接观察原始异常，之后继续运行下一单元。
magazine.title = "新标题"

AttributeError: property 'title' of 'Magazine' object has no setter

## 6 检查属性时避免意外执行 getter

getattr 和 hasattr 都可能触发描述器以及属性访问钩子。只想查看定义时，可使用 inspect.getattr\_static，取得属性的静态对象，不触发这些动态查找过程。

静态查找可能返回 property 或槽描述器自身，不能把结果一律当作属性值；它也不负责生成 \_\_getattr\_\_ 动态提供的属性。本章只检查自己定义的已知名称，不遍历并调用未知对象的属性。

In [12]:
import inspect


class PreviewCounter:
    """用计数显示属性读取是否执行了 getter。"""

    def __init__(self) -> None:
        self.reads = 0

    @property
    def preview(self) -> str:
        """生成预览，并累计实际生成次数。"""
        self.reads += 1
        return "预览内容"


preview_counter = PreviewCounter()
static_preview = inspect.getattr_static(preview_counter, "preview")
print(static_preview is vars(PreviewCounter)["preview"])  # True：原 property。
print(preview_counter.reads)  # 0：静态读取没有运行 getter。
print(preview_counter.preview)  # 预览内容：这次明确执行已知 getter。
print(preview_counter.reads)  # 1：普通读取产生了实际副作用。

True
0
预览内容
1


## 7 slots 声明实例的存储位置

\_\_slots\_\_ 可以声明实例字段。对于没有从父类继承实例字典、也没有显式声明字典的类，它阻止自动创建实例的 \_\_dict\_\_，从而限制未声明字段的动态赋值。

槽（slot）本身由类上的描述器实现。它限制可保存的字段，不会自动校验字段类型，也不表示对象不可变。槽字段应在初始化方法中赋值，不能在同一个类体中再用同名类属性设置默认值。

前面的 Percent 使用 \_score 等存储属性。如果与 slots 配合，需要为这些实际存储名称预留槽位；给公开描述器名称再声明同名槽会冲突。

In [13]:
class SlottedPoint:
    """只保存两个坐标，坐标单位由调用者约定。"""

    __slots__ = ("x", "y")

    def __init__(self, x: float, y: float) -> None:
        self.x = x
        self.y = y


point = SlottedPoint(1.0, 2.0)
point.x = 3.0
print(point.x, point.y)  # 3.0 2.0：已有槽仍然可写。
print(hasattr(point, "__dict__"))  # False：这是本章已知且无动态钩子的类。
print(inspect.getattr_static(point, "x") is vars(SlottedPoint)["x"])
# True：静态查看返回槽描述器，不是坐标值 3.0。

3.0 2.0
False
True


In [14]:
# 预期 AttributeError：直接观察原始异常，之后继续运行下一单元。
point.label = "原点附近"

AttributeError: 'SlottedPoint' object has no attribute 'label'

### 7.1 继承与弱引用的边界

子类不声明 \_\_slots\_\_，通常又会得到实例字典和弱引用支持。子类声明空元组，可以不增加槽；已有的父类槽仍然可用。若父类本来就有实例字典，子类声明 slots 不能移除它。

没有继承弱引用支持时，使用 slots 的类需要在声明中加入 \_\_weakref\_\_，实例才能被弱引用。需要动态属性时可以显式加入 \_\_dict\_\_。弱引用只观察对象，不承担保持对象存活的责任。

不要重复声明父类已有槽名。多个带槽父类参与多重继承时，最多一个父类可以贡献非空槽布局，否则可能产生布局冲突并抛出 TypeError。

In [15]:
import weakref


class OpenPoint(SlottedPoint):
    """未声明 slots，因此恢复动态属性。"""


# 空 slots 沿用已有槽位，但不另加实例字典；下一个类单独增加弱引用槽。
class CompactPoint(SlottedPoint):
    """保留父类槽，不增加实例字典。"""

    __slots__ = ()


class WeakPoint(SlottedPoint):
    """在父类槽的基础上增加弱引用支持。"""

    __slots__ = ("__weakref__",)


class SlottedRecord(Record):
    """空槽不能消除普通父类带来的实例字典。"""

    __slots__ = ()


open_point = OpenPoint(1.0, 2.0)
open_point.label = "可扩展"
print(vars(open_point))  # {'label': '可扩展'}：坐标仍保存在父类槽中。
print(hasattr(CompactPoint(1.0, 2.0), "__dict__"))  # False。
print(hasattr(SlottedRecord(), "__dict__"))  # True：字典来自父类。

{'label': '可扩展'}
False
True


In [16]:
# 预期 TypeError：直接观察原始异常，之后继续运行下一单元。
weakref.ref(point)

TypeError: cannot create weak reference to 'SlottedPoint' object

In [17]:
weak_point = WeakPoint(1.0, 2.0)
point_reference = weakref.ref(weak_point)
print(point_reference() is weak_point)  # True：新增的弱引用槽有效。

True


## 8 \_\_new\_\_ 创建实例，\_\_init\_\_ 初始化实例

通常调用一个类时，\_\_new\_\_ 先创建并返回实例，\_\_init\_\_ 再初始化这个实例，且必须返回 None。\_\_new\_\_ 的首个参数 cls 是请求创建实例的类，不需要显式添加 staticmethod。

不可变类型的子类常在 \_\_new\_\_ 中确定对象值。下面先规范化字符串，再让 str 的创建方法产生子类实例；\_\_init\_\_ 只保存原输入，不能把已创建字符串的内容原地改掉。

如果 \_\_new\_\_ 返回的不是请求类的实例，常规构造流程不会对该返回对象调用请求类的 \_\_init\_\_。普通可变对象通常只需要实现 \_\_init\_\_。

In [18]:
class NormalizedCode(str):
    """创建去除首尾空白并转成大写的字符串子类实例。"""

    def __new__(cls, value: str) -> "NormalizedCode":
        if not isinstance(value, str):
            raise TypeError("代码必须是字符串")
        normalized = value.strip().upper()
        return super().__new__(cls, normalized)

    def __init__(self, value: str) -> None:
        self.original = value


course_code = NormalizedCode("  py31  ")
print(course_code)  # PY31：字符串值在 __new__ 中确定。
print(repr(course_code.original))  # '  py31  '：__init__ 接收原始参数。
print(type(course_code) is NormalizedCode)  # True：创建了请求的子类实例。
print(isinstance(course_code, str))  # True：仍然可以作为字符串使用。
course_code.original = "附加说明"
print(course_code)  # PY31：修改附加属性没有改变字符串内容。

PY31
'  py31  '
True
True
PY31


## 9 type 也能创建类

type 传一个对象时返回它的类型；传三个参数时创建新类。写法 type(name, bases, namespace) 中，name 是类名字符串，bases 是基类元组，namespace 是包含属性和方法的字典；bases 为空时使用 object 作为基类。

普通 class 定义通常由 type 创建类对象，因此普通类本身也是 type 的实例。下面把现成函数放入类命名空间，随后通过实例访问时，仍会发生方法绑定；不需要拼接并执行源码。

In [19]:
def describe_kind(self) -> str:
    """读取绑定实例所使用的分类。"""
    return "分类：" + self.kind


class_namespace = {
    "__module__": __name__,
    "kind": "笔记",
    "describe": describe_kind,
}
DynamicRecord = type("DynamicRecord", (object,), class_namespace)
dynamic_record = DynamicRecord()
print(dynamic_record.describe())  # 分类：笔记；函数成为绑定方法。
print(type(dynamic_record) is DynamicRecord)  # True：实例的类型是新类。
print(type(DynamicRecord) is type)  # True：新类自身由 type 创建。
class_namespace["kind"] = "字典后来修改"
print(DynamicRecord.kind)  # 笔记：type 已复制用于创建类的命名空间。

分类：笔记
True
True
笔记


## 10 用 \_\_init\_subclass\_\_ 初始化后续子类

如果需求是在子类创建后设置约定，可以先考虑 \_\_init\_subclass\_\_。它定义在父类中，却以新子类作为 cls 接收调用，并会被隐式当作类方法处理；不必先编写元类。

类定义行中的自定义关键字可以传给这个钩子。实现应取走自己需要的参数，把剩余参数通过 super 继续传递，支持协作继承。最终的 object.\_\_init\_subclass\_\_ 不接受多余参数；metaclass 关键字由类创建机制处理，不传给这个钩子。

下面的 Document 要求后续子类声明 kind。这个约定在类定义时执行，不会在每次创建文档实例时重复执行。

In [20]:
class Document:
    """为后续子类记录文档分类。"""

    kind = "plain"

    def __init_subclass__(
        cls, *, kind: str, **kwargs: object
    ) -> None:
        super().__init_subclass__(**kwargs)
        cls.kind = kind
        print("init_subclass：", cls.__name__)

    def describe(self) -> str:
        """返回当前文档分类。"""
        return self.kind


class MarkdownDocument(Document, kind="markdown"):
    """在类定义时声明分类。"""


# 上面的类定义产生一次 init_subclass 日志。
print(MarkdownDocument().describe())  # markdown：创建实例不再调用该钩子。
print(MarkdownDocument().describe())  # markdown：第二个实例仍无钩子日志。
print(Document.kind)  # plain：初始化的是新子类，不是父类。

init_subclass： MarkdownDocument
markdown
markdown
plain


## 11 元类参与类创建过程

元类（metaclass）是创建类的类。继承 type 并作为 metaclass 参数传入，可以定制类的创建过程；后续子类通常继承这个元类。

| 阶段 | 中文名称／含义 |
| --- | --- |
| \_\_prepare\_\_ | 在类体执行前准备命名空间，应定义为类方法 |
| class body | 类体在准备好的命名空间中执行 |
| metaclass.\_\_new\_\_ | 接收类名、基类与命名空间，创建类对象 |
| metaclass.\_\_init\_\_ | 初始化已经创建的类对象 |

调用 type.\_\_new\_\_ 时，类成员的 \_\_set\_name\_\_ 先执行，然后调用父类的 \_\_init\_subclass\_\_。因此子类钩子已经能使用完成名称绑定的描述器。这些发生在元类 \_\_new\_\_ 返回之前。

CPython 3.12 会在需要时把隐式的 \_\_class\_\_ 闭包单元放入命名空间的 \_\_classcell\_\_ 项，供零参数 super 使用。元类转交或复制命名空间时必须保留它；过滤掉这项会破坏类创建。下面原样转交命名空间。

In [21]:
class TraceMeta(type):
    """显示类创建各阶段，不改变传入的类命名空间。"""

    @classmethod
    def __prepare__(
        mcls, name: str, bases: tuple[type, ...], **kwargs: object
    ) -> dict[str, object]:
        print("prepare：", name)
        return super().__prepare__(name, bases, **kwargs)

    def __new__(
        mcls,
        name: str,
        bases: tuple[type, ...],
        namespace: dict[str, object],
        **kwargs: object,
    ) -> type:
        print("new 开始，含 classcell：", "__classcell__" in namespace)
        # 完整转交类体命名空间，保留零参数 super 所需的 __classcell__。
        created_class = super().__new__(
            mcls, name, bases, namespace, **kwargs
        )
        print("new 完成：", name)
        return created_class

    def __init__(
        cls,
        name: str,
        bases: tuple[type, ...],
        namespace: dict[str, object],
        **kwargs: object,
    ) -> None:
        super().__init__(name, bases, namespace, **kwargs)
        print("meta init：", name)


class TracedDocument(Document, metaclass=TraceMeta, kind="trace"):
    """保留零参数 super 所需的类闭包。"""

    print("class body：TracedDocument")

    def describe(self) -> str:
        """在父类分类后增加追踪标记。"""
        return super().describe() + " / traced"


# 依次观察 prepare、class body、new 开始、init_subclass、new 完成、meta init。
# classcell 检查为 True；类体中的 super 需要对应的闭包单元。
print(TracedDocument().describe())  # trace / traced：零参数 super 正常工作。
print(type(TracedDocument) is TraceMeta)  # True：元类负责创建类对象。

prepare： TracedDocument
class body：TracedDocument
new 开始，含 classcell： True
init_subclass： TracedDocument
new 完成： TracedDocument
meta init： TracedDocument
trace / traced
True


## 12 类装饰器在类创建完成后应用

类装饰器接收已经创建的类，其返回值再绑定到类名。它适合对明确选择的类追加属性或做转换；元类则能更早参与命名空间准备和类对象创建。

装饰器不会因继承而自动再调用一次，但它写入的普通类属性仍可能被子类继承。\_\_init\_subclass\_\_ 则专门作用于后续子类。多个基类的元类必须兼容，否则类定义会抛出 TypeError；不要只为设置一个标签就引入元类。

In [22]:
def mark_training(cls: type) -> type:
    """为指定类添加教学用途标记。"""
    cls.training = True
    print("decorator：", cls.__name__)
    return cls


@mark_training
class TrainingDocument(TracedDocument, kind="training"):
    """类创建结束后再添加教学标记。"""


# 先出现父类所用元类的各阶段，最后才出现 decorator 日志。
print(TrainingDocument.training)  # True：类装饰器已处理该类。


class DerivedTraining(TrainingDocument, kind="child"):
    """继承类属性，但不自动重新应用类装饰器。"""


# 子类会执行元类与子类钩子，不出现 DerivedTraining 的 decorator 日志。
print(DerivedTraining.training)  # True：标记作为普通类属性被继承。

prepare： TrainingDocument
new 开始，含 classcell： False
init_subclass： TrainingDocument
new 完成： TrainingDocument
meta init： TrainingDocument
decorator： TrainingDocument
True
prepare： DerivedTraining
new 开始，含 classcell： False
init_subclass： DerivedTraining
new 完成： DerivedTraining
meta init： DerivedTraining
True


### 12.1 只需要转换值时使用普通函数

工具应对应需要改变的对象：处理一次输入值，可以写普通函数；复用属性读写规则，可以写描述器；约束后续子类，可以使用 \_\_init\_subclass\_\_。只有确实需要参与类创建过程时，再考虑元类。

例如，前面的 NormalizedCode 演示了不可变子类的创建机制。如果实际需求只是把课程代码规范化，返回普通字符串的函数已经足够。

In [23]:
def normalize_course_code(value: str) -> str:
    """返回规范化后的课程代码，不引入新的对象类型。"""
    if not isinstance(value, str):
        raise TypeError("代码必须是字符串")
    return value.strip().upper()


normalized_code = normalize_course_code("  py31  ")
print(normalized_code)  # PY31：本次值转换已经完成。
print(type(normalized_code) is str)  # True：调用方得到普通字符串。

PY31
True


## 本章小结

（1）先沿 MRO 确定同名类成员，再比较描述器与实例字典的优先级；缺失回退由普通访问流程衔接。

（2）描述器管理属性读写，值应保存在各实例中。property 是数据描述器，普通方法来自非数据描述器。

（3）slots 限制存储字段，仍要考虑父类字典、子类声明和弱引用支持；它不等于类型校验或不可变。

（4）\_\_new\_\_ 创建实例，\_\_init\_\_ 初始化实例。创建类也有阶段；优先选择足够简单的钩子或函数，并在元类转交命名空间时保留 \_\_classcell\_\_。

自查：能否解释一次属性读取为何得到该值，以及某条初始化逻辑是在定义类时还是创建实例时执行？

## 练习

（1）先预测下面三行输出，再执行核对。解释为什么两个实例字典项对读取的影响不同，以及删除 render 项后发生了什么。核对标准是预测与输出一致，并能指出数据描述器、非数据描述器各自的位置。

In [24]:
exercise_magazine = Magazine("练习提纲")
vars(exercise_magazine)["title"] = "字典标题"
exercise_magazine.render = lambda: "练习预览"
print(exercise_magazine.title)
print(exercise_magazine.render())
del exercise_magazine.render
print(exercise_magazine.render())
# 先记录预测，运行后结合实例字典和类中的定义核对。

练习提纲
练习预览
正文：练习提纲


（2）继承 Percent 编写 CountField，只接受 0 到 100 的内置整数，表示完成项数；不接受 bool、浮点数或字符串。重写 \_\_set\_\_ 完成类型限制，再复用父类的范围检查。

建立含 count 字段的类，检查 0、100 可写入，-1、101 抛出 ValueError，True、False、40.0、"40" 抛出 TypeError。每次失败后旧值不变；两个实例分别保存自己的值，通过类名访问时仍能取得描述器。

In [25]:
exercise_valid_counts = (0, 100)
exercise_invalid_counts = (-1, 101, True, False, 40.0, "40")
# 在此实现 CountField 和使用它的类，逐项检查边界与实例隔离。
# 预期异常只捕获对应类型，未抛出时应让检查失败。

（3）继承 TraceMeta 编写 TaggedMeta，在 \_\_new\_\_ 中复制整个 namespace，加入 category="course"，再把副本和所有类关键字转交父类实现。不要删除 \_\_classcell\_\_。

用这个元类创建 Document 的子类，传入 kind="tagged"；重写 describe，通过零参数 super 取得父类结果并追加 " / checked"。再应用 mark\_training 装饰器。检查实例描述为 "tagged / checked"，类属性 category 为 "course"、training 为 True，且装饰器日志出现在元类初始化之后。

In [26]:
# 在此实现 TaggedMeta 和使用它的文档子类，再验证描述与两个类属性。
# 保留整个命名空间，并让父类继续处理 kind 等关键字。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| Python 官方文档（3.12） | 属性与描述器：[默认实例读取与 MRO](https://docs.python.org/3.12/howto/descriptor.html#invocation-from-an-instance)、[描述器协议与两种优先级](https://docs.python.org/3.12/howto/descriptor.html#descriptor-protocol)、[没有 \_\_get\_\_ 时的边界](https://docs.python.org/3.12/reference/datamodel.html#invoking-descriptors)、[\_\_getattribute\_\_](https://docs.python.org/3.12/reference/datamodel.html#object.__getattribute__)、[\_\_getattr\_\_ 与 getter 失败](https://docs.python.org/3.12/reference/datamodel.html#object.__getattr__)、[名称通知](https://docs.python.org/3.12/howto/descriptor.html#automatic-name-notification)、[按字段名保存实例值](https://docs.python.org/3.12/howto/descriptor.html#customized-names)、[类访问的 owner 参数](https://docs.python.org/3.12/howto/descriptor.html#invocation-from-a-class)、[property](https://docs.python.org/3.12/howto/descriptor.html#properties)、[函数与绑定方法](https://docs.python.org/3.12/howto/descriptor.html#functions-and-methods)。校验与检查：[bool 的继承关系](https://docs.python.org/3.12/library/stdtypes.html#boolean-type-bool)、[isinstance](https://docs.python.org/3.12/library/functions.html#isinstance)、[float 与非有限值](https://docs.python.org/3.12/library/functions.html#float)、[NaN 的比较](https://docs.python.org/3.12/reference/expressions.html#value-comparisons)、[vars](https://docs.python.org/3.12/library/functions.html#vars)、[静态属性读取](https://docs.python.org/3.12/library/inspect.html#fetching-attributes-statically)。实例与类创建：[slots 及继承限制](https://docs.python.org/3.12/reference/datamodel.html#slots)、[弱引用](https://docs.python.org/3.12/library/weakref.html#weakref.ref)、[\_\_new\_\_](https://docs.python.org/3.12/reference/datamodel.html#object.__new__)、[\_\_init\_\_](https://docs.python.org/3.12/reference/datamodel.html#object.__init__)、[str.strip](https://docs.python.org/3.12/library/stdtypes.html#str.strip)、[str.upper](https://docs.python.org/3.12/library/stdtypes.html#str.upper)、[type 的两种用法](https://docs.python.org/3.12/library/functions.html#type)、[子类钩子与名称通知](https://docs.python.org/3.12/reference/datamodel.html#customizing-class-creation)、[元类选择与冲突](https://docs.python.org/3.12/reference/datamodel.html#determining-the-appropriate-metaclass)、[准备命名空间](https://docs.python.org/3.12/reference/datamodel.html#preparing-the-class-namespace)、[执行类体](https://docs.python.org/3.12/reference/datamodel.html#executing-the-class-body)、[创建类对象、钩子顺序与 \_\_classcell\_\_](https://docs.python.org/3.12/reference/datamodel.html#creating-the-class-object)、[类装饰器的应用](https://docs.python.org/3.12/reference/compound_stmts.html#class-definitions)。 |
| Python PEP（peps.python.org） | [PEP 487 提案中的两个钩子](https://peps.python.org/pep-0487/#proposal)、[减少元类冲突的动机](https://peps.python.org/pep-0487/#key-benefits)、[名称通知先于子类初始化](https://peps.python.org/pep-0487/#implementation-details)。提案说明设计动机；本章运行语义以 Python 3.12 文档为准。 |